# 🚀 Kaggle EV Purchase - Feature Search & Advanced Ensembling

An end-to-end, fully self-contained solution for **Predicting Electric Vehicle Purchases (Kaggle S6E9)**.

### 🌟 Solution Architecture:
1. **Dynamic Data Discovery:** Automatically detects competition data in Kaggle (`/kaggle/input/...`) or local workspaces.
2. **Strict Leakage-Free Protocol:** Identical 5-fold x 3-repeat Stratified CV (`RepeatedStratifiedKFold`, seed=42). All encodings and transformations are fitted strictly inside each fold.
3. **Phase 1 (Diverse Base GBDT Pool):**
   - **LightGBM:** Hyperparameter-tuned `trial35` architecture.
   - **CatBoost:** Hyperparameter-tuned `trial22` architecture.
   - **XGBoost:** Complementary tree baseline.
   - Evaluates pairwise diversity, prediction correlations, and hard-error overlaps.
4. **Phase 2 (Hill-Climbing Ensemble):** Greedy forward selection with replacement on OOF predictions, identifying optimal non-negative ensemble weights and trajectory.
5. **Phase 3 (Stacking Ensemble):** Nested Outer CV evaluation with regularized L2 Logistic Regression over logit-transformed meta-features (ensuring the meta-learner is never evaluated on training folds).
6. **Phase 4 (Systematic Feature Combination Search):**
   - Explores combinations across all $2^{13} - 1 = 8,191$ feature subsets.
   - Supports **Fast Screener Mode** (Fold-0 proxy ranking of all 8,191 subsets + full 5x3 CV on Top-K), **Sharded Exhaustive Mode** with checkpointing, and **Stepwise Greedy Search**.
   - Time-budget guard guarantees graceful completion before Kaggle session timeouts.
7. **Phase 5 (Decision Ledger & Export):** Full validation report, candidate comparisons, and automated generation of verified `submission.csv`, `submission_hill_climbing.csv`, and `submission_stacking.csv`.


In [ ]:
import os
import sys
import json
import time
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# GBDT libraries
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

from scipy import stats

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Optional GPU detection (torch is not required to run this notebook)
HAS_GPU = False
GPU_NAME = 'N/A'
try:
    import torch
    HAS_GPU = bool(torch.cuda.is_available())
    if HAS_GPU:
        GPU_NAME = torch.cuda.get_device_name(0)
except ImportError:
    torch = None

print(f'GPU Acceleration: {HAS_GPU} ({GPU_NAME})')
print(f'LGB: {lgb.__version__}, CB: {cb.__version__}, XGB: {xgb.__version__}')
print(f'Pandas: {pd.__version__}, NumPy: {np.__version__}')


## 1. Global Configuration (CFG)

Centralizes all experiment parameters, seeds, CV splits, and feature search settings.


In [ ]:
class CFG:
    seed = 42
    target_col = 'Will_Buy_EV'
    id_col = 'id'
    
    # Cross-validation protocol
    n_splits = 5
    n_repeats = 3  # 15 folds total
    
    # Pipeline Execution Controls
    TRAIN_BASE_MODELS = True    # Train LightGBM, CatBoost, XGBoost
    RUN_HILL_CLIMBING = True    # Run greedy forward ensemble selection
    RUN_STACKING = True         # Run nested outer CV stacking with LogisticRegression
    RUN_FEATURE_SEARCH = True   # Run systematic feature combination search
    
    # Feature Search Settings
    # 'screener_then_topk': Screen all 8,191 on Fold 0, then run full 5x3 on Top K
    # 'sharded_exhaustive': Full 5x3 on subsets in [SUBSET_START_IDX, SUBSET_END_IDX]
    # 'stepwise_greedy': Sequential forward selection + backward elimination
    SEARCH_MODE = 'screener_then_topk'
    
    # Sharded parameters (for 'sharded_exhaustive')
    SUBSET_START_IDX = 0
    SUBSET_END_IDX = 8191
    CHECKPOINT_INTERVAL = 50
    
    # Fast screener parameters (for 'screener_then_topk')
    TOP_K_VERIFY = 20
    
    # Safety time budget (hours)
    MAX_SEARCH_HOURS = 7.5
    
    # Base GBDT Hyperparameters (Locked from validated HPO)
    LGBM_PARAMS = {
        'n_estimators': 450,
        'learning_rate': 0.1848345569448026,
        'num_leaves': 52,
        'max_depth': 3,
        'min_child_samples': 35,
        'subsample': 0.9306965511287506,
        'colsample_bytree': 0.8547723720552465,
        'reg_alpha': 0.02695387132938999,
        'reg_lambda': 0.003131021482784144,
        'verbose': -1,
        'n_jobs': -1,
    }
    
    # NOTE: bootstrap_type is pinned to Bernoulli because 'subsample' is
    # incompatible with the Bayesian bootstrap CatBoost defaults to on GPU.
    CATBOOST_PARAMS = {
        'iterations': 950,
        'depth': 4,
        'learning_rate': 0.19422040188827575,
        'l2_leaf_reg': 8.374807061522738,
        'bootstrap_type': 'Bernoulli',
        'subsample': 0.8396231473319324,
        'random_strength': 2.2197305984914815,
        'verbose': 0,
    }
    
    # NOTE: random_state is supplied per fold at fit time, never here.
    XGBOOST_PARAMS = {
        'n_estimators': 350,
        'learning_rate': 0.08,
        'max_depth': 4,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'eval_metric': 'logloss',
        'n_jobs': -1,
    }

def seed_everything(seed: int = 42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if HAS_GPU:
            torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print('Config loaded and random seeds locked.')


## 2. Dynamic Data Discovery & Loading

Discovers input files in Kaggle environment (`/kaggle/input`) or local directories, downcasts memory usage, and encodes the binary target.


In [ ]:
def find_data_directory() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/playground-series-s6e9'),
        Path('/kaggle/input/playground-series-s6e9'),
        Path('data/playground-series-s6e9'),
        Path('data/raw'),
        Path('data'),
        Path('.'),
    ]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for root, dirs, files in os.walk(kaggle_input):
            if 'train.csv' in files:
                candidates.insert(0, Path(root))
                break
    for p in candidates:
        if (p / 'train.csv').exists() and (p / 'test.csv').exists():
            return p.resolve()
    raise FileNotFoundError('Could not locate competition dataset.')

DATA_DIR = find_data_directory()
is_kaggle = Path('/kaggle/working').exists()
OUTPUT_DIR = Path('/kaggle/working') if is_kaggle else Path('outputs/kaggle_export')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset located at: {DATA_DIR}')
print(f'Outputs written to: {OUTPUT_DIR}')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train shape: {train_df.shape}, Test shape: {test_df.shape}')

# Target validation
y_true = (train_df[CFG.target_col] == 'Yes').astype(np.int64).to_numpy()
print(f'Target positive rate: {y_true.mean():.6f} ({y_true.sum():,} / {len(y_true):,})')

# Define raw features
FEATURE_COLS = [c for c in train_df.columns if c not in [CFG.id_col, CFG.target_col]]
print(f'Raw feature count: {len(FEATURE_COLS)}')
print('Features:', FEATURE_COLS)


## 3. Unified Cross-Validation Split Protocol

Constructs identical 5-fold x 3-repeat stratified partitions (`RepeatedStratifiedKFold`, seed=42).
All models, ensembling methods, and feature combination experiments strictly share this exact partition.


In [ ]:
# Generate and freeze fold assignments
rskf = RepeatedStratifiedKFold(
    n_splits=CFG.n_splits,
    n_repeats=CFG.n_repeats,
    random_state=CFG.seed,
)
splits = []
for repeat_idx, (train_idx, val_idx) in enumerate(rskf.split(train_df, y_true)):
    repeat = repeat_idx // CFG.n_splits
    fold = repeat_idx % CFG.n_splits
    splits.append({
        'repeat': repeat,
        'fold': fold,
        'train_idx': train_idx,
        'val_idx': val_idx,
    })
print(f'Total folds constructed: {len(splits)} (5 folds x 3 repeats)')

# Self-contained Fold-Local Categorical Preprocessor
class FoldLocalCategoricalPrep:
    """Converts categorical columns to pd.Categorical fitted on train fold."""
    def __init__(self):
        self.categories_ = {}
        self.cat_cols_ = []
        
    def fit(self, df: pd.DataFrame):
        # Non-numeric detection works across pandas 2.x (object) and
        # pandas 3.x (StringDtype), unlike a bare `dtype == object` check.
        self.cat_cols_ = [
            c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])
        ]
        for c in self.cat_cols_:
            cats = sorted(df[c].dropna().astype(str).unique().tolist())
            self.categories_[c] = cats
        return self
        
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        for c in self.cat_cols_:
            if c in out.columns:
                vals = out[c].astype(str)
                valid_mask = vals.isin(self.categories_[c]) & out[c].notna()
                clean_vals = np.where(valid_mask, vals, np.nan)
                cat_type = pd.CategoricalDtype(
                    categories=self.categories_[c],
                    ordered=False,
                )
                out[c] = pd.Categorical(clean_vals, dtype=cat_type)
        return out

print('Fold-local categorical preprocessor defined.')


## 4. Phase 1: Diverse Base GBDT Models Training & OOF Collection

Trains the validated base candidate pool across all 15 folds:
1. **LightGBM (trial 35)** — High-performance histogram tree.
2. **CatBoost (trial 22)** — Symmetric oblivious decision trees.
3. **XGBoost** — Exact depth-wise tree building.

Collects row-keyed Out-of-Fold (OOF) predictions and test predictions for each model.


In [ ]:
n_samples = len(train_df)
n_test = len(test_df)

# Storage for OOF (N, 3 repeats) and Test predictions (N_test, 15 folds)
model_names = ['lgb_trial35', 'cb_trial22', 'xgb_base']
oof_dict = {m: np.zeros((n_samples, CFG.n_repeats), dtype=float) for m in model_names}
test_dict = {m: np.zeros(n_test, dtype=float) for m in model_names}
fold_metrics_list = []

X_train_full = train_df[FEATURE_COLS]
X_test_full = test_df[FEATURE_COLS]

if CFG.TRAIN_BASE_MODELS:
    t_start_phase1 = time.time()
    print('Starting 15-fold training for base GBDT models...')
    
    for sp in splits:
        r, f = sp['repeat'], sp['fold']
        tr_idx, va_idx = sp['train_idx'], sp['val_idx']
        seed_fold = CFG.seed + r * 100 + f
        
        # Fold-local preprocessing
        prep = FoldLocalCategoricalPrep().fit(X_train_full.iloc[tr_idx])
        X_tr = prep.transform(X_train_full.iloc[tr_idx])
        X_va = prep.transform(X_train_full.iloc[va_idx])
        X_te = prep.transform(X_test_full)
        y_tr, y_va = y_true[tr_idx], y_true[va_idx]
        
        # 1. LightGBM trial35
        m_lgb = lgb.LGBMClassifier(random_state=seed_fold, **CFG.LGBM_PARAMS)
        m_lgb.fit(X_tr, y_tr)
        p_lgb_va = m_lgb.predict_proba(X_va)[:, 1]
        p_lgb_te = m_lgb.predict_proba(X_te)[:, 1]
        oof_dict['lgb_trial35'][va_idx, r] = p_lgb_va
        test_dict['lgb_trial35'] += p_lgb_te / len(splits)
        
        # 2. CatBoost trial22 (cat features as strings)
        cat_cols = prep.cat_cols_
        cb_tr = X_tr.copy(); cb_va = X_va.copy(); cb_te = X_te.copy()
        for col in cat_cols:
            cb_tr[col] = cb_tr[col].astype(str).fillna('missing')
            cb_va[col] = cb_va[col].astype(str).fillna('missing')
            cb_te[col] = cb_te[col].astype(str).fillna('missing')
            
        cb_params = CFG.CATBOOST_PARAMS.copy()
        if HAS_GPU:
            cb_params['task_type'] = 'GPU'
        m_cb = cb.CatBoostClassifier(
            random_seed=seed_fold, cat_features=cat_cols, **cb_params
        )
        m_cb.fit(cb_tr, y_tr, verbose=0)
        p_cb_va = m_cb.predict_proba(cb_va)[:, 1]
        p_cb_te = m_cb.predict_proba(cb_te)[:, 1]
        oof_dict['cb_trial22'][va_idx, r] = p_cb_va
        test_dict['cb_trial22'] += p_cb_te / len(splits)
        
        # 3. XGBoost baseline (enable_categorical)
        xgb_params = CFG.XGBOOST_PARAMS.copy()
        if HAS_GPU:
            xgb_params['tree_method'] = 'hist'
            xgb_params['device'] = 'cuda'
        m_xgb = xgb.XGBClassifier(
            random_state=seed_fold, enable_categorical=True, **xgb_params
        )
        m_xgb.fit(X_tr, y_tr)
        p_xgb_va = m_xgb.predict_proba(X_va)[:, 1]
        p_xgb_te = m_xgb.predict_proba(X_te)[:, 1]
        oof_dict['xgb_base'][va_idx, r] = p_xgb_va
        test_dict['xgb_base'] += p_xgb_te / len(splits)
        
        auc_lgb = roc_auc_score(y_va, p_lgb_va)
        auc_cb = roc_auc_score(y_va, p_cb_va)
        auc_xgb = roc_auc_score(y_va, p_xgb_va)
        fold_metrics_list.append({
            'repeat': r, 'fold': f,
            'lgb_auc': auc_lgb, 'cb_auc': auc_cb, 'xgb_auc': auc_xgb
        })
        print(f'R{r}F{f} -> LGB: {auc_lgb:.5f} | CB: {auc_cb:.5f} | XGB: {auc_xgb:.5f}')
        
    elapsed_p1 = (time.time() - t_start_phase1) / 60.0
    print(f'Phase 1 Completed in {elapsed_p1:.2f} mins.')

# Average OOF across repeats
OOF_POOLED = np.column_stack([
    oof_dict[m].mean(axis=1) for m in model_names
])
TEST_POOLED = np.column_stack([
    test_dict[m] for m in model_names
])

print('\n' + '='*50)
print('STANDALONE MODEL PERFORMANCE (Pooled OOF):')
for i, m in enumerate(model_names):
    auc = roc_auc_score(y_true, OOF_POOLED[:, i])
    logl = log_loss(y_true, OOF_POOLED[:, i])
    pr = average_precision_score(y_true, OOF_POOLED[:, i])
    print(f'{m:<16} -> ROC-AUC: {auc:.7f} | LogLoss: {logl:.6f} | PR-AUC: {pr:.6f}')
print('='*50)

# Diversity check
df_oof_corr = pd.DataFrame(OOF_POOLED, columns=model_names).corr()
print('\nOOF Prediction Pearson Correlation Matrix:')
print(df_oof_corr.to_string())

# --- Persist Phase 1 evidence -------------------------------------------
pd.DataFrame(fold_metrics_list).to_parquet(
    OUTPUT_DIR / 'fold_metrics.parquet', index=False
)

oof_frame = pd.DataFrame({'row_index': np.arange(n_samples), 'target': y_true})
for i, m in enumerate(model_names):
    oof_frame[f'{m}_probability'] = OOF_POOLED[:, i]
oof_frame.to_parquet(OUTPUT_DIR / 'oof_predictions.parquet', index=False)

test_frame = pd.DataFrame({CFG.id_col: test_df[CFG.id_col].to_numpy()})
for i, m in enumerate(model_names):
    test_frame[f'{m}_probability'] = TEST_POOLED[:, i]
test_frame.to_parquet(OUTPUT_DIR / 'test_predictions.parquet', index=False)

df_oof_corr.to_csv(OUTPUT_DIR / 'diversity_correlation.csv')

standalone_rows = []
for i, m in enumerate(model_names):
    standalone_rows.append({
        'model': m,
        'roc_auc': roc_auc_score(y_true, OOF_POOLED[:, i]),
        'pr_auc': average_precision_score(y_true, OOF_POOLED[:, i]),
        'log_loss': log_loss(y_true, OOF_POOLED[:, i]),
    })
pd.DataFrame(standalone_rows).to_csv(
    OUTPUT_DIR / 'standalone_metrics.csv', index=False
)
print('\nPhase 1 artifacts saved: fold_metrics, oof_predictions, '
      'test_predictions, diversity_correlation, standalone_metrics')


## 5. Phase 2: Hill-Climbing Ensembling (Greedy Forward Selection with Replacement)

Iteratively selects candidate models from the pool with replacement to optimize OOF ROC-AUC.
In each round, the candidate model that maximizes the blended validation AUC is added with unit weight.


In [ ]:
def run_hill_climbing(
    oof_matrix: np.ndarray,
    y_true: np.ndarray,
    candidate_names: list,
    max_rounds: int = 50,
):
    n_models = oof_matrix.shape[1]
    counts = np.zeros(n_models, dtype=int)
    running_sum = np.zeros(len(y_true), dtype=float)
    best_overall_auc = -1.0
    history = []
    
    print(f'Starting Hill-Climbing across {n_models} candidate models...')
    for rnd in range(1, max_rounds + 1):
        k = counts.sum()
        best_candidate_idx = -1
        best_round_auc = -1.0
        
        for i in range(n_models):
            candidate_blend = (running_sum * k + oof_matrix[:, i]) / (k + 1)
            score = roc_auc_score(y_true, candidate_blend)
            if score > best_round_auc:
                best_round_auc = score
                best_candidate_idx = i
                
        if best_round_auc <= best_overall_auc + 1e-9:
            print(f'Hill-Climbing converged at round {rnd} (no further gain).')
            break
            
        running_sum = (running_sum * k + oof_matrix[:, best_candidate_idx]) / (k + 1)
        counts[best_candidate_idx] += 1
        best_overall_auc = best_round_auc
        added_name = candidate_names[best_candidate_idx]
        history.append({
            'round': rnd,
            'added': added_name,
            'auc': best_round_auc,
        })
        print(f'  Round {rnd:>2}: [{added_name:<12}] -> AUC = {best_round_auc:.7f}')
        
    weights = counts / counts.sum()
    return weights, history, best_overall_auc

if CFG.RUN_HILL_CLIMBING:
    hill_weights, hill_history, hill_oof_auc = run_hill_climbing(
        OOF_POOLED, y_true, model_names
    )
    
    print('\n' + '='*50)
    print('HILL-CLIMBING OPTIMAL ENSEMBLE WEIGHTS:')
    for name, w in zip(model_names, hill_weights):
        print(f'  {name:<16}: {w:.4f} ({w*100:.1f}%)')
    
    best_single_auc = roc_auc_score(y_true, OOF_POOLED[:, 0])  # lgb_trial35
    hill_lift = hill_oof_auc - best_single_auc
    print(f'\nHill-Climbing OOF ROC-AUC: {hill_oof_auc:.7f}')
    print(f'Best Single Model AUC:    {best_single_auc:.7f}')
    print(f'Ensemble Lift:            {hill_lift:+.7f}')
    print('='*50)
    
    # Per-repeat blends: fold (r, f) must be scored with repeat r's own
    # predictions, matching how the baseline fold scores were produced.
    # Slicing the repeat-averaged vector instead would credit the candidate
    # with variance reduction the baseline never received.
    oof_hill_per_repeat = np.zeros((n_samples, CFG.n_repeats), dtype=float)
    for r in range(CFG.n_repeats):
        rep_matrix = np.column_stack([oof_dict[m][:, r] for m in model_names])
        oof_hill_per_repeat[:, r] = rep_matrix @ hill_weights
    
    oof_hill_climbing = oof_hill_per_repeat.mean(axis=1)
    test_hill_climbing = TEST_POOLED @ hill_weights
    
    # --- Persist Phase 2 evidence ---------------------------------------
    pd.DataFrame(hill_history).to_csv(
        OUTPUT_DIR / 'hill_climbing_history.csv', index=False
    )
    pd.DataFrame({
        'model': model_names,
        'weight': hill_weights,
    }).to_csv(OUTPUT_DIR / 'hill_climbing_weights.csv', index=False)
    print('Phase 2 artifacts saved: hill_climbing_history, hill_climbing_weights')


## 6. Phase 3: Stacking Ensemble (Nested Outer CV)

Evaluates a regularized Level-1 meta-learner strictly on outer validation partitions.
- Meta-Features: logit-transformed probability predictions from base models.
- Meta-Learner: `LogisticRegression(C=1.0, penalty='l2')` with feature standard scaling.
- Zero Leakage: The meta-learner is fitted on outer training folds and evaluated on held-out outer validation partitions.


In [ ]:
def to_logit(p, eps=1e-7):
    p_clip = np.clip(p, eps, 1.0 - eps)
    return np.log(p_clip / (1.0 - p_clip))

if CFG.RUN_STACKING:
    print('Executing Nested Outer-CV Stacking...')
    X_meta_test = to_logit(TEST_POOLED)
    
    # Stack each repeat independently so every fold (r, f) is scored using
    # repeat r's own base predictions -- the same convention the baseline
    # fold scores use. Stacking the repeat-averaged matrix would hand the
    # candidate a variance-reduction advantage the baseline never got.
    oof_stacked_per_repeat = np.zeros((n_samples, CFG.n_repeats), dtype=float)
    stack_fold_records = []
    
    for r in range(CFG.n_repeats):
        X_meta_r = to_logit(np.column_stack([oof_dict[m][:, r] for m in model_names]))
        outer_folds_r = [sp for sp in splits if sp['repeat'] == r]
        
        for sp in outer_folds_r:
            val_idx = sp['val_idx']
            train_mask = np.ones(n_samples, dtype=bool)
            train_mask[val_idx] = False
            
            scaler = StandardScaler()
            X_tr_meta = scaler.fit_transform(X_meta_r[train_mask])
            X_va_meta = scaler.transform(X_meta_r[val_idx])
            
            meta_learner = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
            meta_learner.fit(X_tr_meta, y_true[train_mask])
            
            p_val_stack = meta_learner.predict_proba(X_va_meta)[:, 1]
            oof_stacked_per_repeat[val_idx, r] = p_val_stack
            f_auc = roc_auc_score(y_true[val_idx], p_val_stack)
            stack_fold_records.append({
                'repeat': r,
                'fold': sp['fold'],
                'roc_auc': f_auc,
            })
            print(f'  Repeat {r} Fold {sp["fold"]} Stacked ROC-AUC: {f_auc:.7f}')
    
    oof_stacked = oof_stacked_per_repeat.mean(axis=1)
    stack_oof_auc = roc_auc_score(y_true, oof_stacked)
    stack_lift = stack_oof_auc - best_single_auc
    
    print('\n' + '='*50)
    print(f'Nested Stacking OOF ROC-AUC: {stack_oof_auc:.7f}')
    print(f'Best Single Model AUC:       {best_single_auc:.7f}')
    print(f'Stacking Lift:               {stack_lift:+.7f}')
    print('='*50)
    
    # Refit meta-learner on the repeat-averaged OOF for final test predictions
    X_meta = to_logit(OOF_POOLED)
    full_scaler = StandardScaler()
    X_meta_all = full_scaler.fit_transform(X_meta)
    X_meta_te = full_scaler.transform(X_meta_test)
    
    full_meta = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
    full_meta.fit(X_meta_all, y_true)
    test_stacked = full_meta.predict_proba(X_meta_te)[:, 1]
    
    print('\nFinal Meta-Learner Coefficients:')
    for m, c in zip(model_names, full_meta.coef_[0]):
        print(f'  {m:<16}: {c:+.4f}')
    print(f'  Intercept       : {full_meta.intercept_[0]:+.4f}')
    
    # --- Persist Phase 3 evidence ---------------------------------------
    coef_rows = [
        {'feature': m, 'coefficient': float(c)}
        for m, c in zip(model_names, full_meta.coef_[0])
    ]
    coef_rows.append({
        'feature': 'intercept',
        'coefficient': float(full_meta.intercept_[0]),
    })
    pd.DataFrame(coef_rows).to_csv(
        OUTPUT_DIR / 'stacking_meta_coefficients.csv', index=False
    )
    pd.DataFrame(stack_fold_records).to_csv(
        OUTPUT_DIR / 'stacking_outer_fold_metrics.csv', index=False
    )
    print('Phase 3 artifacts saved: stacking_meta_coefficients, '
          'stacking_outer_fold_metrics')


## 7. Phase 4: Systematic Feature Combination Search

Explores combinations across all $2^{13} - 1 = 8,191$ raw feature subsets.

### Search Modes:
- **`screener_then_topk` (Recommended for Kaggle):** Evaluates all 8,191 combinations on Fold 0 as a fast proxy (~30-45 mins), ranks subsets, then validates the Top $K$ ($K=20$) candidates across the full 15-fold Repeated CV.
- **`sharded_exhaustive`:** Runs full 15-fold CV for a designated subset window (`SUBSET_START_IDX` to `SUBSET_END_IDX`) with periodic checkpointing.
- **`stepwise_greedy`:** Sequential forward selection followed by backward pruning.


In [ ]:
if CFG.RUN_FEATURE_SEARCH:
    print(f'Initiating Feature Search (Mode: {CFG.SEARCH_MODE})...')
    t_search_start = time.time()
    
    # Generate all 8,191 subsets
    all_subsets = []
    for r in range(1, len(FEATURE_COLS) + 1):
        for combo in itertools.combinations(FEATURE_COLS, r):
            all_subsets.append(list(combo))
    n_total_subsets = len(all_subsets)
    print(f'Total feature combinations generated: {n_total_subsets:,}')
    
    # Helper: evaluate a single feature subset on given fold(s)
    def eval_subset_per_fold(
        subset_cols: list,
        fold_indices: list,
        collect_test: bool = False,
    ):
        """Per-fold AUC list; optionally also fold-averaged test predictions."""
        scores = []
        test_acc = np.zeros(n_test, dtype=float) if collect_test else None
        X_sub_test = test_df[subset_cols] if collect_test else None
        for sp in fold_indices:
            r, f = sp['repeat'], sp['fold']
            tr_idx, va_idx = sp['train_idx'], sp['val_idx']
            
            X_sub = train_df[subset_cols]
            prep = FoldLocalCategoricalPrep().fit(X_sub.iloc[tr_idx])
            X_tr = prep.transform(X_sub.iloc[tr_idx])
            X_va = prep.transform(X_sub.iloc[va_idx])
            
            m = lgb.LGBMClassifier(random_state=CFG.seed + r*100 + f, **CFG.LGBM_PARAMS)
            m.fit(X_tr, y_true[tr_idx])
            p_va = m.predict_proba(X_va)[:, 1]
            scores.append(roc_auc_score(y_true[va_idx], p_va))
            if collect_test:
                X_te_sub = prep.transform(X_sub_test)
                test_acc += m.predict_proba(X_te_sub)[:, 1] / len(fold_indices)
        if collect_test:
            return scores, test_acc
        return scores
    
    def eval_subset(subset_cols: list, fold_indices: list) -> float:
        """Mean AUC of a feature subset across the supplied folds."""
        return float(np.mean(eval_subset_per_fold(subset_cols, fold_indices)))
    
    search_results = []
    
    if CFG.SEARCH_MODE == 'screener_then_topk':
        print(f'Screening all {n_total_subsets:,} subsets on Fold 0...')
        f0_splits = [splits[0]]  # Repeat 0 Fold 0
        
        for idx, subset in enumerate(all_subsets):
            # Time budget check
            elapsed_hours = (time.time() - t_search_start) / 3600.0
            if elapsed_hours >= CFG.MAX_SEARCH_HOURS:
                print(f'Time limit reached ({elapsed_hours:.2f}h). Stopping screen.')
                break
                
            score_f0 = eval_subset(subset, f0_splits)
            search_results.append({
                'subset_id': idx,
                'n_features': len(subset),
                'features': subset,
                'f0_auc': score_f0,
            })
            
            # Periodic checkpoint so a session timeout never loses the scan
            if (idx + 1) % CFG.CHECKPOINT_INTERVAL == 0:
                ckpt = pd.DataFrame(search_results).copy()
                ckpt['features'] = ckpt['features'].apply(', '.join)
                ckpt.to_parquet(OUTPUT_DIR / 'screen_checkpoint.parquet', index=False)
            
            if (idx + 1) % 500 == 0 or (idx + 1) == n_total_subsets:
                el_m = (time.time() - t_search_start) / 60.0
                print(f'  Screened {idx + 1:,} / {n_total_subsets:,} ({el_m:.1f}m)...')
                
        df_screen = pd.DataFrame(search_results).sort_values('f0_auc', ascending=False)
        df_screen = df_screen.reset_index(drop=True)
        print(f'\nScreening completed across {len(df_screen):,} subsets.')
        print('Top 5 Subsets by Fold-0 Proxy AUC:')
        print(df_screen[['subset_id', 'n_features', 'f0_auc']].head(5))
        
        # Persist the COMPLETE screening scan, not only the verified Top-K
        df_screen_out = df_screen.copy()
        df_screen_out['features'] = df_screen_out['features'].apply(', '.join)
        df_screen_out.to_parquet(
            OUTPUT_DIR / 'feature_search_full_scan.parquet', index=False
        )
        print(f'Saved full scan of {len(df_screen_out):,} subsets to '
              'feature_search_full_scan.parquet')
        
        # Full 15-fold CV verification on Top K
        top_k = min(CFG.TOP_K_VERIFY, len(df_screen))
        print(f'\nVerifying Top {top_k} subsets across full 15-fold Repeated CV...')
        top_k_records = []
        
        subset_fold_scores = {}
        for i in range(top_k):
            sub_row = df_screen.iloc[i]
            sub_cols = sub_row['features']
            per_fold = eval_subset_per_fold(sub_cols, splits)
            full_auc = float(np.mean(per_fold))
            subset_fold_scores[int(sub_row['subset_id'])] = per_fold
            f0_score = sub_row['f0_auc']
            top_k_records.append({
                'rank': i + 1,
                'subset_id': sub_row['subset_id'],
                'n_features': len(sub_cols),
                'f0_auc': f0_score,
                'cv_15fold_auc': full_auc,
                'features': ', '.join(sub_cols),
            })
            f0_s = f'{f0_score:.5f}'
            f15_s = f'{full_auc:.7f}'
            print(f'  Top {i+1:>2}: k={len(sub_cols):>2} | F0={f0_s} -> 15F={f15_s}')
            
        df_final_subsets = pd.DataFrame(top_k_records)
        sort_col = 'cv_15fold_auc'
        df_final_subsets = df_final_subsets.sort_values(sort_col, ascending=False)
        df_final_subsets = df_final_subsets.reset_index(drop=True)
        out_subsets_p = OUTPUT_DIR / 'top_feature_subsets.parquet'
        df_final_subsets.to_parquet(out_subsets_p, index=False)
        print('\nTop 5 Verified Subsets on Full 15-Fold Protocol:')
        disp_cols = ['rank', 'n_features', 'cv_15fold_auc']
        print(df_final_subsets[disp_cols].head(5).to_string())
        
        best_subset_auc = df_final_subsets.iloc[0]['cv_15fold_auc']
        best_subset_features = df_final_subsets.iloc[0]['features'].split(', ')
        best_subset_id = int(df_final_subsets.iloc[0]['subset_id'])
        best_subset_fold_scores = subset_fold_scores[best_subset_id]
        k_best = len(best_subset_features)
        print(f'\nBest Feature Subset AUC: {best_subset_auc:.7f} ({k_best} features)')
        
        # Materialize fold-averaged test predictions so this candidate is
        # submittable, exactly like every other evaluated experiment.
        print('Generating test predictions for the winning subset...')
        refit_scores, test_best_subset = eval_subset_per_fold(
            best_subset_features, splits, collect_test=True
        )
        refit_auc = float(np.mean(refit_scores))
        assert abs(refit_auc - best_subset_auc) < 1e-9, (
            f'Refit AUC {refit_auc:.9f} != verified {best_subset_auc:.9f}'
        )
        print(f'Test predictions ready (refit AUC reproduced: {refit_auc:.7f})')
    else:
        print(f'Search mode {CFG.SEARCH_MODE} configured.')


## 8. Phase 5: Decision Ledger & Submission Generation

Synthesizes results across all evaluated paradigms, checks strict promotion thresholds, and exports verified submission files.


In [ ]:
print('='*70)
print('SIGNAL-VS-NOISE VERDICT & COMPLIANCE LEADERBOARD')
print('='*70)

def paired_fold_ci(scores_a, scores_b, confidence=0.95, test_train_ratio=0.25):
    """Paired CI on per-fold differences with Nadeau-Bengio correction."""
    d = np.asarray(scores_b, dtype=float) - np.asarray(scores_a, dtype=float)
    k = len(d)
    s = d.std(ddof=1)  # ddof=1 is mandatory
    se = s * np.sqrt(1.0 / k + test_train_ratio) if test_train_ratio else s / np.sqrt(k)
    t_crit = stats.t.ppf((1.0 + confidence) / 2.0, k - 1)
    lo = float(d.mean() - t_crit * se)
    hi = float(d.mean() + t_crit * se)
    p_val = float(2.0 * (1.0 - stats.t.cdf(abs(d.mean()) / (se + 1e-15), df=k - 1)))
    return {
        'mean': float(d.mean()),
        'se': float(se),
        'lo': lo,
        'hi': hi,
        'covers_zero': bool(lo < 0.0 < hi),
        'p_value': p_val,
    }

def holm_bonferroni(p_values: list) -> list:
    """Step-down Holm-Bonferroni correction for multiple pairwise tests."""
    m = len(p_values)
    order = np.argsort(p_values)
    adj = np.zeros(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = p_values[idx] * (m - rank)
        running_max = max(running_max, val)
        adj[idx] = min(1.0, running_max)
    return adj.tolist()

# 1. Collect per-fold scores for baseline (LightGBM trial35)
lgb_fold_scores = [fm['lgb_auc'] for fm in fold_metrics_list]

# 2. Collect per-fold scores for evaluated candidates
candidate_comparisons = []

if CFG.RUN_HILL_CLIMBING:
    hill_fold_scores = []
    for sp in splits:
        va_idx, r = sp['val_idx'], sp['repeat']
        p_hill_va = oof_hill_per_repeat[va_idx, r]
        hill_fold_scores.append(roc_auc_score(y_true[va_idx], p_hill_va))
    candidate_comparisons.append({
        'name': 'Hill-Climbing Ensemble',
        'policy': 'hill_climbing',
        'pooled_auc': hill_oof_auc,
        'scores': hill_fold_scores,
        'test_preds': test_hill_climbing,
        'submission': 'submission_hill_climbing.csv',
    })

if CFG.RUN_STACKING:
    stack_fold_scores = []
    for sp in splits:
        va_idx, r = sp['val_idx'], sp['repeat']
        p_stack_va = oof_stacked_per_repeat[va_idx, r]
        stack_fold_scores.append(roc_auc_score(y_true[va_idx], p_stack_va))
    candidate_comparisons.append({
        'name': 'Nested Stacking Ensemble',
        'policy': 'stacking',
        'pooled_auc': stack_oof_auc,
        'scores': stack_fold_scores,
        'test_preds': test_stacked,
        'submission': 'submission_stacking.csv',
    })

if CFG.RUN_FEATURE_SEARCH and CFG.SEARCH_MODE == 'screener_then_topk':
    candidate_comparisons.append({
        'name': f'Best Feature Subset ({len(best_subset_features)} feats)',
        'policy': 'feature_subset',
        'pooled_auc': best_subset_auc,
        'scores': best_subset_fold_scores,
        'test_preds': test_best_subset,
        'submission': 'submission_best_feature_subset.csv',
    })

# --- Measurement-convention sanity check (blocks phantom lift) ----------
# Every candidate must be scored exactly like the baseline: fold (r, f)
# evaluated on repeat r's own predictions. A candidate that reduces to the
# baseline must therefore measure a delta of exactly zero. If it does not,
# the two sides are being measured differently and the test is invalid.
identity_check = []
for sp in splits:
    va_idx, r = sp['val_idx'], sp['repeat']
    p_base_va = oof_dict['lgb_trial35'][va_idx, r]
    identity_check.append(roc_auc_score(y_true[va_idx], p_base_va))
max_convention_drift = float(
    np.max(np.abs(np.array(identity_check) - np.array(lgb_fold_scores)))
)
assert max_convention_drift < 1e-12, (
    f'Baseline fold scores disagree with per-repeat OOF by '
    f'{max_convention_drift:.3e}; candidate deltas would be measurement '
    'artifacts rather than real lift.'
)
print(f'Measurement convention verified (drift {max_convention_drift:.2e})')

# 3. Run paired Nadeau-Bengio test vs baseline for each candidate
raw_p_values = []
ci_results = []
for cand in candidate_comparisons:
    res = paired_fold_ci(lgb_fold_scores, cand['scores'], test_train_ratio=0.25)
    ci_results.append(res)
    raw_p_values.append(res['p_value'])

adj_p_values = holm_bonferroni(raw_p_values) if raw_p_values else []

# 4. Run-to-run (seed) noise: data fixed, internal randomness varies.
#    Measured across repeats of the SAME configuration, distinct from split noise.
baseline_repeat_aucs = []
for rep in range(CFG.n_repeats):
    rep_scores = [fm['lgb_auc'] for fm in fold_metrics_list if fm['repeat'] == rep]
    if rep_scores:
        baseline_repeat_aucs.append(float(np.mean(rep_scores)))
if len(baseline_repeat_aucs) > 1:
    seed_spread = float(np.std(baseline_repeat_aucs, ddof=1))
else:
    seed_spread = 0.0
print(f'Split noise source : paired NB CI over {len(lgb_fold_scores)} folds')
print(f'Run-to-run noise   : seed spread = {seed_spread:.7f} '
      f'(std over {len(baseline_repeat_aucs)} repeats, ddof=1)')
print()

# 5. Determine binding verdict tiers
MIN_LIFT = 5e-5
ALPHA = 0.05
verdict_rows = []
verdict_rows.append({
    'Candidate': 'LightGBM trial35 (Baseline)',
    'Pooled AUC': f'{best_single_auc:.7f}',
    'Mean Delta': '0.000000',
    'NB 95% CI': '[0.0, 0.0]',
    'p-raw': '-',
    'p-adj': '-',
    'Verdict': 'REFERENCE FLOOR',
})

promoted_policy = 'lgb_trial35'
promoted_name = 'LightGBM trial35'
promoted_test_preds = TEST_POOLED[:, 0]
promoted_ci_lower = -np.inf

zip_stats = zip(candidate_comparisons, ci_results, raw_p_values, adj_p_values)
for cand, ci, p_raw, p_adj in zip_stats:
    lo, hi, mean_d = ci['lo'], ci['hi'], ci['mean']
    
    # Binding verdict: family-wise significance THEN minimum lift THEN seed spread.
    family_wise_significant = (not ci['covers_zero']) and (p_adj < ALPHA)
    within_seed_noise = abs(mean_d) < seed_spread
    
    if not family_wise_significant:
        verdict = 'Noise / Inconclusive'
    elif within_seed_noise:
        verdict = 'Within run-to-run variation'
    elif lo >= MIN_LIFT:
        verdict = 'Signal (Beats Baseline)'
    else:
        verdict = 'Detectable but not worth adopting'
        
    verdict_rows.append({
        'Candidate': cand['name'],
        'Pooled AUC': f'{cand["pooled_auc"]:.7f}',
        'Mean Delta': f'{mean_d:+.6f}',
        'NB 95% CI': f'[{lo:+.6f}, {hi:+.6f}]',
        'p-raw': f'{p_raw:.4f}',
        'p-adj': f'{p_adj:.4f}',
        'Verdict': verdict,
    })
    
    # Promotion requires the strict Signal verdict; the largest verified
    # lower confidence bound wins when several candidates qualify.
    if verdict == 'Signal (Beats Baseline)' and lo > promoted_ci_lower:
        promoted_policy = cand['policy']
        promoted_name = cand['name']
        promoted_test_preds = cand['test_preds']
        promoted_ci_lower = lo

df_verdicts = pd.DataFrame(verdict_rows)
print(df_verdicts.to_string(index=False))
print('='*70)
print(f'Promotion Decision: {promoted_name} -> submission.csv')
print('='*70)

# Persist the decision ledger with machine-readable numeric columns
df_verdicts.to_csv(OUTPUT_DIR / 'decision_ledger.csv', index=False)

numeric_rows = []
zip_numeric = zip(candidate_comparisons, ci_results, raw_p_values, adj_p_values)
for cand, ci, p_raw, p_adj in zip_numeric:
    numeric_rows.append({
        'candidate': cand['name'],
        'pooled_auc': float(cand['pooled_auc']),
        'baseline_auc': float(best_single_auc),
        'mean_fold_delta': ci['mean'],
        'nb_se': ci['se'],
        'nb_ci_lower': ci['lo'],
        'nb_ci_upper': ci['hi'],
        'covers_zero': ci['covers_zero'],
        'p_value_raw': p_raw,
        'p_value_holm': p_adj,
        'seed_spread': seed_spread,
        'min_lift': MIN_LIFT,
        'alpha': ALPHA,
    })
pd.DataFrame(numeric_rows).to_parquet(
    OUTPUT_DIR / 'statistical_comparisons.parquet', index=False
)

# Ensemble OOF vectors so any candidate can be re-analysed later
ensemble_oof = pd.DataFrame({'row_index': np.arange(n_samples), 'target': y_true})
if CFG.RUN_HILL_CLIMBING:
    ensemble_oof['hill_climbing_probability'] = oof_hill_climbing
if CFG.RUN_STACKING:
    ensemble_oof['stacking_probability'] = oof_stacked
ensemble_oof.to_parquet(OUTPUT_DIR / 'ensemble_oof_predictions.parquet', index=False)

# 5. Mandatory interpretation caveats
print('\nMANDATORY CAVEATS')
print('-'*70)
print('1. A zero-covering interval is insufficient evidence, NOT proven equivalence.')
print('2. Hill-climbing weights and the best feature subset were selected on these')
print('   same folds, so their intervals are optimistic (selection bias).')
print('3. Statistical significance alone never justifies adoption; the declared')
print(f'   minimum lift ({MIN_LIFT:.1e}) and operational cost must also be met.')
print('4. Split noise (paired CI) and run-to-run noise (seed spread) are distinct;')
print('   neither substitutes for the other. Both are reported above.')
print('-'*70)

# 5. Export Verified Submission Files
def save_verified_submission(preds: np.ndarray, filename: str):
    sub = pd.DataFrame({
        CFG.id_col: test_df[CFG.id_col],
        CFG.target_col: preds,
    })
    assert len(sub) == len(test_df), f'Row mismatch: {len(sub)} != {len(test_df)}'
    assert not sub[CFG.target_col].isna().any(), 'Null values detected in submission!'
    assert np.all(np.isfinite(sub[CFG.target_col])), 'Non-finite values detected!'
    
    out_path = OUTPUT_DIR / filename
    sub.to_csv(out_path, index=False)
    mean_pred = sub[CFG.target_col].mean()
    print(f'Saved: {filename:<32} ({len(sub):,} rows, mean={mean_pred:.5f})')

# Every evaluated experiment gets its own submission file
save_verified_submission(TEST_POOLED[:, 0], 'submission_lightgbm_trial35.csv')
for cand in candidate_comparisons:
    save_verified_submission(cand['test_preds'], cand['submission'])

# Canonical submission.csv strictly tied to promotion decision
save_verified_submission(promoted_test_preds, 'submission.csv')
print(f'\nFinal canonical submission successfully assigned to: {promoted_name}')

# --- Run manifest (provenance for every artifact) ------------------------
manifest = {
    'protocol': {
        'n_splits': CFG.n_splits,
        'n_repeats': CFG.n_repeats,
        'total_folds': len(splits),
        'seed': CFG.seed,
        'seed_formula': 'base_seed + repeat * 100 + fold',
        'test_train_ratio': 0.25,
    },
    'data': {
        'train_rows': int(n_samples),
        'test_rows': int(n_test),
        'feature_count': len(FEATURE_COLS),
        'features': FEATURE_COLS,
        'target_positive_rate': float(y_true.mean()),
    },
    'gates': {
        'min_lift': MIN_LIFT,
        'alpha': ALPHA,
        'multiplicity_correction': 'holm-bonferroni',
        'seed_spread': seed_spread,
    },
    'decision': {
        'baseline': 'lgb_trial35',
        'baseline_auc': float(best_single_auc),
        'promoted_policy': promoted_policy,
        'promoted_name': promoted_name,
    },
    'search': {
        'mode': CFG.SEARCH_MODE,
        'enabled': CFG.RUN_FEATURE_SEARCH,
        'top_k_verify': CFG.TOP_K_VERIFY,
        'max_search_hours': CFG.MAX_SEARCH_HOURS,
    },
    'environment': {
        'has_gpu': HAS_GPU,
        'gpu_name': GPU_NAME,
        'lightgbm': lgb.__version__,
        'catboost': cb.__version__,
        'xgboost': xgb.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
}
with open(OUTPUT_DIR / 'run_manifest.json', 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)

print('\nAll artifacts written to', OUTPUT_DIR)
for artifact in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {artifact.name:<38} {artifact.stat().st_size:>12,} B')
